In [1]:
# packages import
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import random

This file contains three parts:
1. Train the forecast model for temperature 
2. Train the forecast model for PV output based on temperature
3. Train the forecast model for load curve based on temperature 
4. First generate scenarios of temperature, and generate scenarios for PV output and load curve based on the generated scenarios

In [ ]:
# data reading
data = pd.read_csv(r"solar data_2022.csv", encoding="gbk")
data["hour"] = np.sin(2 * np.pi * data["hour"] / 24)
data

,generation,month,day,hour,Irradiance,Rainfall,RH,SLP,Temp,Vis,WS,WD
0,0.0,1,1,0.000000,6.2852,0.0,83.284,1010.65913,15.720,13.144,2.5708,91.938
1,0.0,1,1,0.258819,6.1749,0.0,82.858,1010.41913,15.440,12.932,2.4456,103.130
2,0.0,1,1,0.500000,6.2855,0.0,83.486,1010.43913,15.270,13.059,3.6489,100.800
3,0.0,1,1,0.707107,6.1753,0.0,83.523,1010.19913,15.071,12.303,2.6138,91.187
4,0.0,1,1,0.866025,5.9553,0.0,85.592,1010.11913,14.881,11.438,2.4305,84.136
...,...,...,...,...,...,...,...,...,...,...,...,...
8704,0.0,12,31,-0.965926,7.7296,0.0,61.192,1009.86913,15.337,15.981,4.2071,353.620
8705,0.0,12,31,-0.866025,7.6190,0.0,64.047,1010.11913,14.475,15.981,3.1189,355.760
8706,0.0,12,31,-0.707107,7.6195,0.0,65.872,1010.17913,14.095,15.979,4.4467,348.860
8707,0.0,12,31,-0.500000,7.8410,0.0,66.376,1010.21913,13.949,15.979,4.6437,351.610


In [3]:
# build windows for 4 hours based on historic 24 hours
def build_windows(df, x_features, y_features, ctx=24, H=4):
    """
        X contains the historical 24 hours' feature data and corresponding prediction output
        y contains the current temperature and next 4 hours
    """
    base_features = y_features + x_features
    arr = df[base_features].values
    X_windows, y_windows = [], []
    for i in range(ctx, len(df)-H):
        curr_X = arr[i-ctx:i, :].reshape(-1)
        X_windows.append(curr_X) # This contains the last 24 hours' historical data
        y_windows.append(arr[i:i+H, 0])
    return np.stack(X_windows), np.stack(y_windows)


## Temperature forecast model

In [4]:
# Data buildup
x_temp_features = ['hour', 'Irradiance', 'Rainfall', 'RH', 'SLP', 'WS', 'WD']
y_temp_feature = ['Temp']
X_temp, y_temp = build_windows(data, x_temp_features, y_temp_feature)
print(y_temp.shape)
X_temp

(8681, 4)


array([[ 1.57200000e+01,  0.00000000e+00,  6.28520000e+00, ...,
         1.01013913e+03,  8.89930000e-01,  7.72770000e+01],
       [ 1.54400000e+01,  2.58819045e-01,  6.17490000e+00, ...,
         1.01001913e+03,  1.69490000e+00,  3.50400000e+02],
       [ 1.52700000e+01,  5.00000000e-01,  6.28550000e+00, ...,
         1.00995913e+03,  1.95200000e+00,  3.47640000e+02],
       ...,
       [ 1.45180000e+01, -9.65925826e-01,  4.05220000e+01, ...,
         1.00957913e+03,  1.23980000e+00,  1.47650000e+01],
       [ 1.38820000e+01, -1.00000000e+00,  9.93790000e+00, ...,
         1.00968913e+03,  2.90120000e+00,  3.51600000e+02],
       [ 1.32550000e+01, -9.65925826e-01,  8.61420000e+00, ...,
         1.00977913e+03,  1.87390000e+00,  3.49280000e+02]],
      shape=(8681, 192))

In [6]:
# packages used for forecast model
import os
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb

In [7]:
# 1) Quantile objective (works with DMatrix in booster API)
def quantile_objective(q):
    def _obj(preds, dtrain):
        y = dtrain.get_label()          # labels from DMatrix
        resid = y - preds
        grad = np.where(resid > 0, -q, 1 - q)
        hess = np.ones_like(preds) * 1e-6
        return grad, hess
    return _obj

# 2) Train a single horizon + quantile model using booster API
def train_quantile_horizon_booster(X_train, y_train_h, q, base_params=None, num_boost_round=800):
    """
    X_train: (N, F)
    y_train_h: (N,)
    q: quantile in [0,1]
    Returns a trained Booster
    """
    dtrain = xgb.DMatrix(X_train, label=y_train_h)
    params = {
        # 'objective': quantile_objective(q),
        'objective': 'reg:quantileerror',
        'quantile_alpha': q,
        'eval_metric': 'mae',
        'max_depth': 6,
        'eta': 0.05,
        'tree_method': 'hist',
        'nthread': -1,
        'seed': 42
    }
    if base_params:
        params.update(base_params)

    booster = xgb.train(params, dtrain, num_boost_round=num_boost_round)
    return booster

# 3) Train for all horizons and quantiles, and save models with manifest
def train_and_save_quantile_models(X, Y, horizons=None, quantiles=[0.1, 0.5, 0.9],
                                   train_ratio=0.8, model_dir='models', base_params=None, num_boost_round=800):
    """
    X: (N, F)
    Y: (N, H) where H is number of horizons (e.g., 4)
    horizons: list of horizons to train (1-based, e.g., [1,2,3,4])
    """
    if horizons is None:
        horizons = list(range(1, Y.shape[1] + 1))

    N = X.shape[0]
    split = int(train_ratio * N)

    X_train, X_valid = X[:split], X[split:]
    Y_train, Y_valid = Y[:split], Y[split:]

    boosters = [[None for _ in quantiles] for _ in horizons]

    manifest_entries = []
    os.makedirs(model_dir, exist_ok=True)

    for hi, h in enumerate(horizons):
        y_tr = Y_train[:, hi]
        y_va = Y_valid[:, hi]

        for qi, q in enumerate(quantiles):
            b = train_quantile_horizon_booster(X_train, y_tr, q, base_params=base_params, num_boost_round=num_boost_round)
            boosters[hi][qi] = b

            path = os.path.join(model_dir, f'h{h}_q{int(q*100)}.model')
            b.save_model(path)
            manifest_entries.append({'horizon': h, 'quantile': q, 'path': path})

    manifest = {
        'horizons': horizons,
        'quantiles': quantiles,
        'train_ratio': train_ratio,
        'model_dir': model_dir,
        'models': manifest_entries
    }

    manifest_path = os.path.join(model_dir, 'manifest.json')
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    return boosters, manifest, X_valid, Y_valid

# 4) Load models from manifest
def load_models_from_manifest(model_dir='models'):
    manifest_path = os.path.join(model_dir, 'manifest.json')
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)

    horizons = manifest['horizons']
    quantiles = manifest['quantiles']

    boosters = []
    for hi, h in enumerate(horizons):
        row = []
        for qi, q in enumerate(quantiles):
            path = manifest['models'][hi * len(quantiles) + qi]['path']
            b = xgb.Booster()
            b.load_model(path)
            row.append(b)
        boosters.append(row)
    return boosters, manifest

# 5) Inference: predict quantiles for new data
def predict_quantiles_from_boosters(X_new, boosters, quantiles=[0.1, 0.5, 0.9]):
    """
    X_new: shape (1, F) or (K, F)
    boosters: list of horizons x quantiles boosters
    Returns: dict horizon -> {quantile: value}
    """
    dtest = xgb.DMatrix(X_new)
    results = {}
    for hi, horizon_boosters in enumerate(boosters):
        h = hi + 1
        row = {}
        for qi, q in enumerate(quantiles):
            preds = horizon_boosters[qi].predict(dtest)
            row[q] = float(preds.ravel()[0])
        results[h] = row
    return results

# 6) Pretty print helper
def print_quantile_forecasts(preds):
    for h in sorted(preds.keys()):
        inner = preds[h]
        print(f"Horizon t+{h}: " + ", ".join([f"q{int(q*100)}={v:.2f}" for q, v in sorted(inner.items())]))

In [ ]:
# Train the forecast model
horizon = [i for i in range(4)]
qs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
temp_models, temp_manifest, X_test, Y_test = train_and_save_quantile_models(X=X_temp, Y=y_temp, horizons=horizon, quantiles=qs, model_dir='temp_models')

temp_models_loaded, temp_manifest_loaded = load_models_from_manifest('temp_models')


In [9]:
X_new = X_temp[0:1, :]
preds = predict_quantiles_from_boosters(X_new, temp_models_loaded, qs)
# Sometimes the quantile regression will produce some crossing quantiles
# We need to write a function to enforce there does not exist 
def enforce_non_crossing(preds, quantiles=[0.1, 0.5, 0.9]):
    # preds: dict horizon -> {quantile: value}
    for h in sorted(preds.keys()):
        # ensure the quantile keys are in ascending order
        vals = [preds[h][q] for q in quantiles]
        # monotone non-decreasing across quantiles
        monotone = []
        cur = vals[0]
        monotone.append(cur)
        for v in vals[1:]:
            cur = max(cur, v)
            monotone.append(cur)
        # write back
        for i, q in enumerate(quantiles):
            preds[h][q] = monotone[i]
    return preds
preds = enforce_non_crossing(preds, qs)
print(y_temp[0])
print(preds[1])
print(preds[2])
print(preds[3])
print(preds[4])

[14.48  14.402 14.279 14.067]
{0.1: 14.187582969665527, 0.2: 14.397235870361328, 0.3: 14.436005592346191, 0.4: 14.502897262573242, 0.5: 14.502897262573242, 0.6: 14.502897262573242, 0.7: 14.502897262573242, 0.8: 14.512263298034668, 0.9: 14.717825889587402}
{0.1: 14.184614181518555, 0.2: 14.34117317199707, 0.3: 14.408662796020508, 0.4: 14.408662796020508, 0.5: 14.408662796020508, 0.6: 14.408662796020508, 0.7: 14.408662796020508, 0.8: 14.408662796020508, 0.9: 15.019375801086426}
{0.1: 14.19082260131836, 0.2: 14.280475616455078, 0.3: 14.280752182006836, 0.4: 14.280752182006836, 0.5: 14.28173828125, 0.6: 14.28173828125, 0.7: 14.28173828125, 0.8: 14.550165176391602, 0.9: 15.293339729309082}
{0.1: 14.070231437683105, 0.2: 14.070231437683105, 0.3: 14.070231437683105, 0.4: 14.073752403259277, 0.5: 14.073752403259277, 0.6: 14.158190727233887, 0.7: 14.158190727233887, 0.8: 14.49850082397461, 0.9: 15.52664852142334}


## PV output forecast model

In [10]:
# data pre-processing
mdays = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
cum_before = np.insert(np.cumsum(mdays), 0, 0)
months = data['month'].to_numpy(dtype=int)
days = data['day'].to_numpy(dtype=int)
data['doy'] = np.sin((cum_before[months-1] + days) * np.pi * 2 / 365)

x_pv_features = ["hour", "Irradiance", "Rainfall", "RH", "SLP", "Temp", "Vis", "WS", "WD", "doy"]
y_pv_feature = ["generation"]
X_pv, y_pv = build_windows(data, x_pv_features, y_pv_feature)
# add a new column of current temperature tp X_pv
curr_temps = [data["Temp"].to_numpy()[24+dd:-4+dd].reshape(-1, 1) for dd in range(4)]
X_pv = np.hstack([X_pv] + curr_temps)
print(y_pv.shape)
X_pv

(8681, 4)


array([[ 0.        ,  0.        ,  6.2852    , ..., 14.402     ,
        14.279     , 14.067     ],
       [ 0.        ,  0.25881905,  6.1749    , ..., 14.279     ,
        14.067     , 14.145     ],
       [ 0.        ,  0.5       ,  6.2855    , ..., 14.067     ,
        14.145     , 13.631     ],
       ...,
       [ 0.805     , -0.96592583, 40.522     , ..., 16.061     ,
        15.337     , 14.475     ],
       [ 0.184     , -1.        ,  9.9379    , ..., 15.337     ,
        14.475     , 14.095     ],
       [ 0.        , -0.96592583,  8.6142    , ..., 14.475     ,
        14.095     , 13.949     ]], shape=(8681, 268))

In [11]:
# Train the forecast model
horizon = [i for i in range(4)]
qs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
pv_models, pv_manifest, X_test, Y_test = train_and_save_quantile_models(X=X_pv, Y=y_pv, horizons=horizon, quantiles=qs, model_dir='pv_models')

pv_models_loaded, pv_manifest_loaded = load_models_from_manifest('pv_models')


/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_30010/299027823.py:68: UserWarning: [14:26:59] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_30010/299027823.py:68: UserWarning: [14:27:03] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_30010/299027823.py:68: UserWarning: [14:27:08] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/

In [12]:
X_new = X_pv[10:11, :]
preds = predict_quantiles_from_boosters(X_new, pv_models_loaded, qs)
preds = enforce_non_crossing(preds, qs)
print(y_pv[10])
print(preds[1])
print(preds[2])
print(preds[3])
print(preds[4])

[ 9.704 13.685 15.78  14.609]
{0.1: 4.808079719543457, 0.2: 7.398871421813965, 0.3: 8.991090774536133, 0.4: 9.736421585083008, 0.5: 9.9141845703125, 0.6: 9.9141845703125, 0.7: 10.118152618408203, 0.8: 10.859326362609863, 0.9: 10.859326362609863}
{0.1: 3.414332389831543, 0.2: 7.968864440917969, 0.3: 11.309920310974121, 0.4: 13.644087791442871, 0.5: 13.644087791442871, 0.6: 13.685267448425293, 0.7: 13.68924617767334, 0.8: 13.909029960632324, 0.9: 14.742467880249023}
{0.1: 3.1100516319274902, 0.2: 8.673957824707031, 0.3: 14.78185749053955, 0.4: 15.797696113586426, 0.5: 15.823657989501953, 0.6: 15.823657989501953, 0.7: 16.052669525146484, 0.8: 16.052669525146484, 0.9: 16.916725158691406}
{0.1: 2.6931004524230957, 0.2: 7.345815658569336, 0.3: 12.143940925598145, 0.4: 13.25393295288086, 0.5: 14.004478454589844, 0.6: 14.611151695251465, 0.7: 14.61628246307373, 0.8: 14.619670867919922, 0.9: 14.932361602783203}


## Load Forecasting Data processing
1. Read data from the supermarket data
2. Do the simple load forecasting based on temperature, hours.
3. Still use XGBoost

In [13]:
load_data = pd.read_csv(r"building data/2zonesupermarket.csv", encoding="gbk")
load_data

,outdoor temperature,load,date,hour
0,1.875,37242.0,0.000000,0.000000
1,0.500,37242.0,0.000000,0.258819
2,-2.250,37242.0,0.000000,0.500000
3,-2.375,37242.0,0.000000,0.707107
4,-3.875,115248.0,0.000000,0.866025
...,...,...,...,...
8755,0.000,181290.0,-0.017213,-0.965926
8756,0.000,181290.0,-0.017213,-0.866025
8757,0.000,55248.0,-0.017213,-0.707107
8758,0.000,37242.0,-0.017213,-0.500000


In [14]:
x_load_features = ['outdoor temperature', 'hour']
y_load_features = ['load']
X_load, y_load = build_windows(load_data, x_load_features, y_load_features)
curr_temps = [load_data['outdoor temperature'].to_numpy()[24+dd:-4+dd].reshape(-1, 1) for dd in range(4)]
X_load = np.hstack([X_load] + curr_temps)
print(y_load.shape)
X_load

(8732, 4)


array([[ 3.72420000e+04,  1.87500000e+00,  0.00000000e+00, ...,
        -3.62500000e+00, -4.62500000e+00, -3.75000000e+00],
       [ 3.72420000e+04,  5.00000000e-01,  2.58819045e-01, ...,
        -4.62500000e+00, -3.75000000e+00, -2.37500000e+00],
       [ 3.72420000e+04, -2.25000000e+00,  5.00000000e-01, ...,
        -3.75000000e+00, -2.37500000e+00, -2.62500000e+00],
       ...,
       [ 2.48520000e+05,  1.75000000e+00, -9.65925826e-01, ...,
         3.75000000e-01,  0.00000000e+00,  0.00000000e+00],
       [ 1.87290000e+05,  1.00000000e+00, -1.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 1.81290000e+05,  1.00000000e+00, -9.65925826e-01, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],
      shape=(8732, 76))

In [15]:
horizon = [i for i in range(4)]
qs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
load_models, load_manifest, X_test, Y_test = train_and_save_quantile_models(X=X_load, Y=y_load, horizons=horizon, quantiles=qs, model_dir='load_models')

load_models_loaded, load_manifest_loaded = load_models_from_manifest('load_models')

/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_30010/299027823.py:68: UserWarning: [14:33:18] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_30010/299027823.py:68: UserWarning: [14:33:19] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/xrsnrkm50550l10wsrzlm0m00000gn/T/ipykernel_30010/299027823.py:68: UserWarning: [14:33:20] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  b.save_model(path)
/var/folders/0r/

In [16]:
X_new = X_load[222:223, :]
preds = predict_quantiles_from_boosters(X_new, load_models_loaded, qs)
preds = enforce_non_crossing(preds, qs)
print(y_load[222])
print(preds[1])
print(preds[2])
print(preds[3])
print(preds[4])

[115248. 241290. 248520. 248520.]
{0.1: 115247.921875, 0.2: 115248.078125, 0.3: 115248.078125, 0.4: 115248.078125, 0.5: 115248.078125, 0.6: 248520.0, 0.7: 248520.0, 0.8: 248520.0, 0.9: 248520.0}
{0.1: 115247.921875, 0.2: 121554.140625, 0.3: 241290.15625, 0.4: 241290.15625, 0.5: 241290.8125, 0.6: 248520.0, 0.7: 248520.0, 0.8: 248520.0, 0.9: 248520.0}
{0.1: 115247.921875, 0.2: 121554.140625, 0.3: 248519.84375, 0.4: 248519.84375, 0.5: 248519.84375, 0.6: 248520.0, 0.7: 248520.0, 0.8: 248520.0, 0.9: 248520.0}
{0.1: 115247.921875, 0.2: 121554.140625, 0.3: 248519.84375, 0.4: 248519.84375, 0.5: 248519.84375, 0.6: 248520.0, 0.7: 248520.0, 0.8: 248520.0, 0.9: 248520.0}
